# 51 Prepare model inputs

<!-- navigation -->
[<kbd>← 21 Subsurface data</kbd>](21%20Subsurface%20data.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>52 Timflow model →</kbd>](52%20Timflow%20model.ipynb)

**Required** · minimal path: 00 → 01 → 10 → 20 → **51** → 52

Selects the model period of one transect, adds the far-field reference and saves the observations for notebook 52.

| | |
|---|---|
| **Input** | transects metadata, piezometers (`project_data.gpkg`), pastastore of the transect |
| **Output** | observations of the model period (`observations_model_period.xlsx` in the transect folder 'timeseries') |

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

from datetime import datetime as dt

import pandas as pd
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
## UPDATE FOR YOUR PROJECT: the transect to work on (a name from the transects metadata)
transect_name = "Papendrecht West"

## 1. Transect metadata
- `polder_level`: water level in the polder [m NAP] (in Dutch: 'polderpeil')
- `undisturbed_head`: undisturbed hydraulic head in the aquifer below the polder [m NAP] (in Dutch: 'onverstoorde stijghoogte')
- `loc_river`: name of the river level series
- `farfield`: the far-field reference of the model (see below)

In [ ]:
meta = project.transect_metadata(transect_name)

polder_level = meta["polder_level"]
undisturbed_head = meta["undisturbed_head"]
loc_river = meta["loc_river"]

meta

## 2. Piezometers
The piezometers of this transect, with their x-coordinate in the model and model layer.

In [ ]:
obslocs = tlt.make_obslocs(project.piezometers(transect_name))
obslocs.sort_values(["x_crosssection", "modellayer"])

## 3. Time series
All series in the pastastore of this transect. Inspect them to choose the model period in step 5: a part with clear tidal fluctuations in the river and the piezometers, without gaps and without disturbances such as pumping or maintenance.

In [ ]:
pstore = project.open_pastastore(transect_name)

tlt.plot_select(pstore, title=transect_name, subset="all series", units=units)
pstore.plots.data_availability("oseries", set_yticks=True, figsize=(10, 5), progressbar=False);

## 4. Far-field reference
The head used as the hydraulic head in the far field (`farfield` in the metadata; change it there or here):
- `"polder_level"`: the water level in the polder. This is the traditional approach of the predecessor tool Watex.
- `"undisturbed_head"`: the undisturbed hydraulic head in the uppermost regional aquifer (Dutch: 'eerste watervoerend pakket'). Its average value is then the reference at large distance.

In [ ]:
loc_farfield = meta["farfield"]  # or "polder_level" / "undisturbed_head"

## 5. Model period
Set the period for the calibration below, based on the figures in step 3. A period of approximately 1 week is recommended: with a longer period, the fitting and calculation may take a long time. The figures below show the selected part, to check the choice.

The time is converted to days since the start of the period. This creates two tables:
- `obs_ref`: the observed hydraulic heads [m NAP], plus the far-field reference as a constant series
- `obs_dh`: the same heads normalized around zero, by subtracting the mean per location

In [ ]:
## UPDATE FOR YOUR TRANSECT: the model period, chosen from the figures in step 3
xlim = [dt(2025, 12, 1), dt(2025, 12, 8)]  # example: the week of the example data

obs_ref = pd.DataFrame()
for name in pstore.oseries_names:
    obs_ref = obs_ref.join(tlt.clip_data(pstore, name, xlim), how="outer")

# Add the far-field reference as a constant series (KeyError for an unknown choice)
farfield_heads = {"polder_level": polder_level, "undisturbed_head": undisturbed_head}
obs_ref[loc_farfield] = farfield_heads[loc_farfield]
print(f"Added '{loc_farfield}' reference at {farfield_heads[loc_farfield]:.3f} {units['head']}")

# Remove empty columns
obs_ref = obs_ref.dropna(axis=1, how="all")

# Heads relative to the average head at each location
obs_dh = obs_ref - obs_ref.mean()

obs_ref.head()

You may want to resample to a fixed time interval. Otherwise, time series with a higher frequency get more weight in the calibration. The count per location:

In [ ]:
obs_ref.describe()

Quick plot of the time series.

_Before plotting, the time series are interpolated to produce a visually correct plot, because they have different intervals._

In [ ]:
savefolder_ts = project.transect_dir(transect_name, "timeseries")

tlt.plot_daily_situation(
    obs_ref, "Daily situation - hydraulic heads",
    name=transect_name, xlim=xlim, savepath=savefolder_ts / "heads_model_period.png", units=units,
)
tlt.plot_daily_situation(
    obs_dh, r"Daily situation - $\Delta h$",
    name=transect_name, xlim=xlim, savepath=savefolder_ts / "heads_model_period_dh.png",
    units=units, relative=True,
);

## 6. Average heads
1. Compared to the reference level [m NAP]
2. Compared to the head at the far-field reference [m]

You may consider using the median instead of the mean, since it is less sensitive to outliers.

In [ ]:
obs_summary = tlt.summarize_observations(obs_ref, obslocs, loc_farfield, loc_river)
obs_summary.style.format(na_rep="-", precision=2)

## 7. Save the observations for the model notebook

In [ ]:
fn_observations = savefolder_ts / "observations_model_period.xlsx"
tlt.write_observations(obs_ref, fn_observations, start=xlim[0], end=xlim[1])  # notebook 52 reads exactly these values
print(f"Saved {project.relpath(fn_observations)} ({xlim[0]:%Y-%m-%d} - {xlim[1]:%Y-%m-%d})")

---

<!-- navigation -->
[<kbd>← 21 Subsurface data</kbd>](21%20Subsurface%20data.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>52 Timflow model →</kbd>](52%20Timflow%20model.ipynb)